In [0]:
-- League - Goals by Period
-- When goals happen: regulation periods and overtime.
-- Change the values in params only; everything below reads from it.
WITH params AS (                              -- change settings here only
  SELECT
    CAST(NULL AS STRING) AS team,  -- team abbreviation; CAST(NULL AS STRING) = every team
    20252026 AS season,            -- e.g. 20242025; CAST(NULL AS INT) = every season
    2 AS game_type                 -- 2 = regular season, 3 = playoffs
),
goals AS (
  SELECT f.period_number, f.period_type
  FROM nhl.gold.fact_event f
  JOIN nhl.gold.dim_game g ON g.game_id = f.game_id
  JOIN nhl.gold.dim_team t ON t.team_id = f.event_owner_team_id
  CROSS JOIN params pr
  WHERE f.type_desc_key = 'goal' AND COALESCE(f.period_type, '') <> 'SO'
    AND g.game_type = pr.game_type
    AND (pr.season IS NULL OR g.season = pr.season)
    AND (pr.team IS NULL OR t.abbrev = pr.team)
)
SELECT CASE WHEN period_type = 'OT' THEN 'OT' ELSE CAST(period_number AS STRING) END AS period,
       COUNT(*)                                          AS goals,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS share_pct
FROM goals
GROUP BY 1
ORDER BY period;